# WhatsApp Chat Analysis & Machine Learning with Spark

This notebook demonstrates a full Big Data pipeline using Apache Spark to analyze WhatsApp chat logs and classify them using Machine Learning.


In [1]:
import os
import sys
import re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Set environment variables for Spark workers
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("WhatsAppNotebook") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

Spark Version: 4.1.1


## 2. Data Loading & Parsing
We use Regular Expressions to parse the standard WhatsApp export format: `[Date, Time] - Sender: Message`.

In [2]:
CHAT_REGEX = r"(\d{2}/\d{2}/\d{4}), (\d{2}:\d{2}) - ([^:]+): (.+)"

def parse_chat_line(line):
    match = re.match(CHAT_REGEX, line)
    if match: return match.groups()
    return (None, None, None, line)

def load_data(base_path):
    all_rows = []
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(".txt"):
                category = os.path.basename(root)
                with open(os.path.join(root, file), "r", encoding="utf-8") as f:
                    for line in f.readlines():
                        if not line.strip(): continue
                        date_str, time_str, sender, message = parse_chat_line(line)
                        if date_str:
                            all_rows.append((category, date_str, time_str, sender, message))
    
    schema = StructType([
        StructField("category", StringType(), True),
        StructField("date", StringType(), True),
        StructField("time", StringType(), True),
        StructField("sender", StringType(), True),
        StructField("message", StringType(), True)
    ])
    return spark.createDataFrame(all_rows, schema)

df = load_data("dataset")
df.show(5)

+--------+----------+-----+---------------+--------------------+
|category|      date| time|         sender|             message|
+--------+----------+-----+---------------+--------------------+
|    work|12/05/2023|09:15|Alice (Manager)|Hey team, we have...|
|    work|12/05/2023|09:17|            Bob|I'm on it! Review...|
|    work|12/05/2023|09:30|        Charlie|Sounds good, I'll...|
|    work|12/05/2023|10:00|            Mom|Don't forget to p...|
|    work|12/05/2023|10:05|            You|Will do! I'll be ...|
+--------+----------+-----+---------------+--------------------+
only showing top 5 rows


## 3. Data Analysis & Insights
Extracting patterns like most active users and peak messaging times.

In [3]:
print("Message count per category:")
df.groupBy("category").count().show()

print("Top 10 Senders:")
df.groupBy("sender").count().sort(F.desc("count")).show(10)

Message count per category:
+--------+-----+
|category|count|
+--------+-----+
|    work|   18|
+--------+-----+

Top 10 Senders:
+------------------+-----+
|            sender|count|
+------------------+-----+
|               You|    4|
|Dave (Best Friend)|    3|
|   Alice (Manager)|    2|
|               Bob|    2|
|           Charlie|    1|
|               Mom|    1|
|    Unknown Number|    1|
|     Netflix Promo|    1|
|               Dad|    1|
|       Scam Caller|    1|
+------------------+-----+
only showing top 10 rows


## 4. Machine Learning Pipeline
We use a Random Forest Classifier to categorize chats based on their content.

In [4]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer, IndexToString
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Pipeline Components
indexer = StringIndexer(inputCol="category", outputCol="label")
tokenizer = Tokenizer(inputCol="message", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=2000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=50)
labelConverter = IndexToString(inputCol="prediction", outputCol="predictedLabel", labels=indexer.fit(df).labels)

pipeline = Pipeline(stages=[indexer, tokenizer, remover, hashingTF, idf, rf, labelConverter])

# Train/Test Split
(train_data, test_data) = df.randomSplit([0.8, 0.2], seed=42)

# Train
model = pipeline.fit(train_data)

# Predict & Evaluate
predictions = model.transform(test_data)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
print(f"Test Accuracy: {evaluator.evaluate(predictions):.4f}")

predictions.select("category", "message", "predictedLabel").show(10)

Test Accuracy: 1.0000
+--------+--------------------+--------------+
|category|             message|predictedLabel|
+--------+--------------------+--------------+
|    work|I'm on it! Review...|          work|
|    work|Hitting legs toda...|          work|
+--------+--------------------+--------------+

